# Parallelization Pattern Hands-On Code Example

The following implementation demonstrates a parallel processing workflow constructed with the LangChain framework. This workflow is designed to execute two independent operations concurrently in response to a single user query. These parallel processes are instantiated as distinct chains or functions, and their respective outputs are subsequently aggregated into a unified result.

In [1]:
# !pip install langchain langchain-community langchain-google-genai langgraph

> Note: Create a `.env` file in the same directory with your Google Generative AI API key:
> ```
> GOOGLE_API_KEY="<your_google_api_key_here>"
> ```

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

In [3]:
load_dotenv(override=True)

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [5]:
# Define independent chains

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
            ("system", "Summarize the following topic concisely: "),
            ("user", "{topic}"),
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
            ("system", "Generate three interesting questions about the following topic: "),
            ("user", "{topic}"),
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
            ("system", "Indentify 5-10 key terms from the following topic, separated by commas: "),
            ("user", "{topic}"),
    ])
    | llm
    | StrOutputParser()
)


In [6]:
# Build the parallel and synthesis chain
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),
    }
)

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Synthesize a comprehensive answer."""),
    ("user", "Original Topic: {topic}"),
])

full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


In [7]:
async def run_parallel_example(topic: str):
    """Asynchronously invokes the parallel processing chain with the specific topic and prints the synthesized result.
    
    Args:
        topic (str): The input topic to be processed by the LangChain chains.
    """
    print(f"\n--- Running Parallel LangChian Example for Topic: '{topic}' ---\n")
    try:
        response = await full_parallel_chain.ainvoke(topic)
        print("\n--- Final response ---\n")
        print(response)
    except Exception as e:
        print(f"Error during chain execution: {e}")

In [8]:
test_topic = "The history of space exploration"
await run_parallel_example(test_topic)


--- Running Parallel LangChian Example for Topic: 'The history of space exploration' ---


--- Final response ---

The history of space exploration is a dynamic narrative of human ambition, technological innovation, and evolving global priorities, beginning in the **mid-20th century** and continuing to redefine humanity's place in the cosmos.

Initially, space exploration was primarily ignited by the intense **Cold War Space Race** between the **Soviet Union and the United States**. This period was driven by a fierce competition for national prestige, ideological supremacy, and technological dominance. The Soviets achieved early, groundbreaking milestones with **Sputnik 1 (1957)**, the first artificial **satellite**, and **Yuri Gagarin (1961)**, the first human in space, demonstrating the power of their **rockets** and pioneering **human spaceflight**. The U.S. responded with the ambitious Apollo program, culminating in the iconic **Apollo 11 mission in 1969**, which saw the **first h

In essence, this code sets up a workflow where multiple LLM calls (for summarizing, questions, and terms) happen at the same time for a given topic, and their results are then combined by a final LLM call. This showcases the core idea of parallelization in an agentic workflow using LangChain.